# Exporting an int8 FinBERT and the Serving Bundle

Notebook 7 of 7. The live app runs on Render's free tier, which allows 512 MB of memory; a full-precision FinBERT (about 440 MB of weights) plus PyTorch would not fit. This notebook exports the headline model to ONNX, quantises its weights to 8-bit integers with ONNX Runtime, measures what that costs on the validation and test splits, and writes the bundle the app loads: the int8 model, its tokenizer, the metrics, chart data and the walk-forward sessions.

The benchmarks are judged on the full-precision model in notebook 4; the int8 numbers here describe the served copy.

## Setup

Installs `onnx` 1.23.2 and `onnxruntime` 1.30.0, the exact versions the app pins, so the int8 kernels here are the ones the app runs. pip output is captured so its progress bars do not reach the log. `SMOKE` evaluates 50 sentences per split; it is off in this version.

In [1]:
import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"

import json
import platform
import shutil
import subprocess
import sys
import time
from pathlib import Path

subprocess.run([sys.executable, "-m", "pip", "install", "--progress-bar", "off", "onnx==1.23.2", "onnxruntime==1.30.0"], check=True, capture_output=True)

import numpy as np
import onnx
import onnxruntime as ort
import pandas as pd
import sklearn
import tokenizers
import torch
import transformers
from onnxruntime.quantization import QuantType, quantize_dynamic
from tokenizers import Tokenizer
from transformers import BertForSequenceClassification

SMOKE = False
MAX_LENGTH = 128
LABELS = ["negative", "neutral", "positive"]
INPUTS = ["input_ids", "attention_mask", "token_type_ids"]
INPUT = Path("/kaggle/input")
OUT = Path("/kaggle/working")
ART = OUT / "artifacts"
TMP = Path("/tmp/export")
ART.mkdir(parents=True, exist_ok=True)
TMP.mkdir(parents=True, exist_ok=True)
PARITY = [
    "Quarterly revenue rose 12 percent, beating analyst expectations.",
    "The company cut its full-year profit forecast after weak holiday sales.",
    "The board will publish the interim report on 14 August.",
    "The lender set aside more money for bad loans as defaults climbed.",
    "Operating margin improved as raw material costs eased.",
    "Shares were little changed after the annual meeting date was confirmed.",
    "Net sales decreased to 210 million euros from 245 million euros a year earlier.",
    "The new contract is expected to add about 40 million dollars to annual sales.",
    "Credit rating agencies downgraded the utility to junk status.",
    "Shareholders approved the merger at an extraordinary general meeting.",
    "Profit before taxes more than doubled thanks to strong demand in Asia.",
    "The retailer plans to close 150 stores and cut 3,000 jobs.",
    "The group runs production plants in Finland, Sweden and Poland.",
    "The dividend will be raised for the tenth consecutive year.",
    "Losses widened as the airline grounded most of its fleet.",
    "The chief executive will step down at the end of the year.",
    "Order intake grew strongly and the backlog reached a record high.",
    "The bank warned that weaker trading income would weigh on second-quarter results.",
    "The company is headquartered in Helsinki.",
    "Investors welcomed a share buyback programme worth 500 million euros.",
]
torch.set_num_threads(4)
transformers.logging.set_verbosity_error()
START = time.time()


def find_file(name):
    hits = sorted(INPUT.rglob(name))
    if not hits:
        raise FileNotFoundError(name)
    return hits[0]


def softmax(logits):
    z = logits.astype(np.float64) - logits.max(axis=-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)


def session(path):
    options = ort.SessionOptions()
    options.intra_op_num_threads = 1
    options.inter_op_num_threads = 1
    return ort.InferenceSession(str(path), options, providers=["CPUExecutionProvider"])


def accuracy(y, p):
    return float(np.mean(y == p))


def macro_f1(y, p):
    cm = np.zeros((3, 3), dtype=np.int64)
    np.add.at(cm, (y, p), 1)
    tp = np.diag(cm)
    denominator = 2 * tp + (cm.sum(axis=0) - tp) + (cm.sum(axis=1) - tp)
    return float(np.divide(2 * tp, denominator, out=np.zeros(3), where=denominator > 0).mean())


def clean(value):
    if isinstance(value, dict):
        return {str(k): clean(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [clean(v) for v in value]
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    if isinstance(value, (np.integer, int)):
        return int(value)
    if isinstance(value, (np.floating, float)):
        return None if not np.isfinite(value) else float(value)
    return value


def write(name, value):
    (ART / name).write_text(json.dumps(clean(value), allow_nan=False), encoding="utf-8")


print(f"smoke={SMOKE}")
print(f"onnx {onnx.__version__}, onnxruntime {ort.__version__}, torch {torch.__version__}, transformers {transformers.__version__}, tokenizers {tokenizers.__version__}")

smoke=False
onnx 1.23.2, onnxruntime 1.30.0, torch 2.11.0+cpu, transformers 5.16.1, tokenizers 0.23.1


**Conclusion.** onnx 1.23.2 and onnxruntime 1.30.0 are installed at exactly the versions the app pins; everything else comes from the CPU image (PyTorch 2.11.0, transformers 5.16.1, tokenizers 0.23.1).

## Inputs

Loads the headline model and its tokenizer from notebook 3, the validation and test sentences from notebook 2, the full model's own predictions on them (notebook 3, chosen seed), and the metrics of notebooks 4 and 6.

In [2]:
model_dir = find_file("model.safetensors").parent
table = pd.read_parquet(find_file("phrasebank.parquet"))
best = json.loads(find_file("best_params.json").read_text(encoding="utf-8"))
fine = pd.read_parquet(find_file("finetune_pred.parquet"))
sentiment = json.loads(find_file("metrics_sentiment.json").read_text(encoding="utf-8"))
market = json.loads(find_file("metrics_market.json").read_text(encoding="utf-8"))
sessions = pd.read_parquet(find_file("sessions.parquet"))
tokenizer = Tokenizer.from_file(str(model_dir / "tokenizer.json"))
tokenizer.enable_truncation(MAX_LENGTH)
tokenizer.no_padding()
splits = {name: table[table["split"] == name].sort_values("sentence_id") for name in ("val", "test")}
if SMOKE:
    splits = {name: rows.iloc[:50] for name, rows in splits.items()}
chosen = fine[fine["seed"] == best["seed"]]
full = {name: chosen[chosen["split"] == name].set_index("sentence_id").loc[rows["sentence_id"], ["p_neg", "p_neu", "p_pos"]].to_numpy().argmax(axis=1) for name, rows in splits.items()}
print(f"headline model: {best['checkpoint']}, seed {best['seed']}; sentences: " + ", ".join(f"{k} {len(v)}" for k, v in splits.items()))
print(f"sessions: {len(sessions)}, of which in the walk-forward test {int(sessions['in_test'].sum())}")

headline model: finbert-pretrain, seed 42; sentences: val 484, test 484
sessions: 566, of which in the walk-forward test 366


**Conclusion.** The headline model (finbert-pretrain, seed 42) is loaded with all 484 validation and 484 test sentences, the full model's own predictions on them from notebook 3, and notebook 6's 566 sessions, 366 of them in the walk-forward test.

## Export to ONNX and a parity check

The model is wrapped so it returns logits only and exported with the TorchScript-based exporter at opset 17, with dynamic batch and sequence axes. The float32 ONNX model is then checked against PyTorch on the test split: the largest logit difference must stay below 1e-4 and the labels must agree. A pass or fail is printed rather than the raw difference, which varies in its last digits between machines.

In [3]:
class Logits(torch.nn.Module):
    def __init__(self, inner):
        super().__init__()
        self.inner = inner

    def forward(self, input_ids, attention_mask, token_type_ids):
        return self.inner(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids).logits


def batch_feed(texts):
    encoded = [tokenizer.encode(t).ids for t in texts]
    width = max(len(e) for e in encoded)
    ids = np.zeros((len(encoded), width), dtype=np.int64)
    mask = np.zeros_like(ids)
    for i, e in enumerate(encoded):
        ids[i, :len(e)] = e
        mask[i, :len(e)] = 1
    return {"input_ids": ids, "attention_mask": mask, "token_type_ids": np.zeros_like(ids)}


model = BertForSequenceClassification.from_pretrained(model_dir, attn_implementation="eager").eval()
FP32 = TMP / "finbert-fp32.onnx"
sample = batch_feed(splits["test"]["sentence"].iloc[:2].tolist())
with torch.no_grad():
    torch.onnx.export(Logits(model).eval(), tuple(torch.from_numpy(sample[k]) for k in INPUTS), str(FP32), input_names=INPUTS, output_names=["logits"],
                      dynamic_axes={**{k: {0: "batch", 1: "sequence"} for k in INPUTS}, "logits": {0: "batch"}}, opset_version=17, dynamo=False)
onnx.checker.check_model(str(FP32))
print(f"model in eval mode after export: {not model.training}")
fp32 = session(FP32)
texts = splits["test"]["sentence"].tolist()
worst, agree = 0.0, 0
for start in range(0, len(texts), 32):
    feed = batch_feed(texts[start:start + 32])
    with torch.no_grad():
        reference = model(**{k: torch.from_numpy(v) for k, v in feed.items()}).logits.numpy()
    got = fp32.run(["logits"], feed)[0]
    worst = max(worst, float(np.abs(reference - got).max()))
    agree += int((reference.argmax(axis=1) == got.argmax(axis=1)).sum())
single_worst, single_agree = 0.0, 0
for text in texts:
    feed = batch_feed([text])
    with torch.no_grad():
        reference = model(**{k: torch.from_numpy(v) for k, v in feed.items()}).logits.numpy()
    got = fp32.run(["logits"], feed)[0]
    single_worst = max(single_worst, float(np.abs(reference - got).max()))
    single_agree += int(reference.argmax() == got.argmax())
print(f"float32 ONNX against PyTorch on {len(texts)} test sentences in padded batches: largest logit difference below 1e-4: {worst < 1e-4}; identical labels {agree} of {len(texts)}")
print(f"float32 ONNX against PyTorch one sentence at a time, as the app scores: largest logit difference below 1e-4: {single_worst < 1e-4}; identical labels {single_agree} of {len(texts)}")
if SMOKE:
    print(f"raw largest logit differences: padded batches {worst:.3e}, one at a time {single_worst:.3e}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/tmp/ipykernel_16/3758618639.py:25: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(Logits(model).eval(), tuple(torch.from_numpy(sample[k]) for k in INPUTS), str(FP32), input_names=INPUTS, output_names=["logits"],
/usr/local/lib/python3.13/dist-packages/transformers/masking_utils.py:212: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if (padding_length := kv_length + kv_offset - attention_mask.shape[-1]) > 0:
/usr/local/lib/

model in eval mode after export: True
float32 ONNX against PyTorch on 484 test sentences in padded batches: largest logit difference below 1e-4: True; identical labels 484 of 484
float32 ONNX against PyTorch one sentence at a time, as the app scores: largest logit difference below 1e-4: True; identical labels 484 of 484


**Conclusion.** The model stays in eval mode after export, and the float32 ONNX graph reproduces PyTorch to within 1e-4 in every logit on all 484 test sentences, in padded batches and one sentence at a time, with identical labels throughout. The eval-mode line is printed because the export wrapper starts in training mode and the exporter restores that mode afterwards; an early smoke run compared against a PyTorch reference with dropout still on and showed gaps above one logit, which were the reference's fault, not the graph's.

## Dynamic int8 quantisation

`quantize_dynamic` stores the weights of every MatMul and the embedding tables (Gather) as unsigned 8-bit integers and quantises activations on the fly. Unsigned weights avoid the saturation that signed weights can hit on CPUs without VNNI instructions, so the results do not depend on which x86 instructions a machine has. The cell prints both file sizes and the inputs of the quantised graph.

In [4]:
INT8 = ART / "finbert-int8.onnx"
quantize_dynamic(str(FP32), str(INT8), op_types_to_quantize=["MatMul", "Gather"], weight_type=QuantType.QUInt8, per_channel=False, reduce_range=False)
int8 = session(INT8)
names = [i.name for i in int8.get_inputs()]
print(f"float32 ONNX {FP32.stat().st_size / 1e6:.1f} MB, int8 ONNX {INT8.stat().st_size / 1e6:.1f} MB")
print(f"int8 graph inputs: {names}")

float32 ONNX 439.2 MB, int8 ONNX 110.5 MB
int8 graph inputs: ['input_ids', 'attention_mask', 'token_type_ids']


**Conclusion.** Dynamic int8 quantisation shrinks the model from 439.2 MB to 110.5 MB, a quarter of its size and small enough to load inside Render's 512 MB with room for the app. The quantised graph keeps the inputs input_ids, attention_mask and token_type_ids.

## What int8 costs

Dynamic quantisation computes one activation scale per tensor, so a sentence scored inside a batch could come out slightly differently from the same sentence scored alone. The app therefore scores every text on its own, and so does this cell. It reports accuracy and macro F1 of the int8 model next to the full model's on both splits, the number of sentences whose label changed, and single-thread latency.

In [5]:
def single_feed(text):
    ids = np.array([tokenizer.encode(text).ids], dtype=np.int64)
    feed = {"input_ids": ids, "attention_mask": np.ones_like(ids), "token_type_ids": np.zeros_like(ids)}
    return {k: feed[k] for k in names}


def int8_probabilities(texts):
    return np.array([softmax(int8.run(["logits"], single_feed(t))[0])[0] for t in texts])


int8_result = {}
for name, rows in splits.items():
    y = rows["label"].to_numpy()
    pred = int8_probabilities(rows["sentence"].tolist()).argmax(axis=1)
    int8_result[name] = {"accuracy": accuracy(y, pred), "macro_f1": macro_f1(y, pred), "full_accuracy": accuracy(y, full[name]), "full_macro_f1": macro_f1(y, full[name]), "flipped": int((pred != full[name]).sum()), "n": len(y)}
    r = int8_result[name]
    print(f"{name}: int8 accuracy {r['accuracy']:.4f} against full {r['full_accuracy']:.4f}; macro F1 {r['macro_f1']:.4f} against {r['full_macro_f1']:.4f}; labels changed {r['flipped']} of {r['n']}")
times = []
for text in splits["test"]["sentence"].iloc[:100]:
    began = time.perf_counter()
    int8.run(["logits"], single_feed(text))
    times.append(time.perf_counter() - began)
print(f"int8 single-thread latency in seconds per sentence: median {np.median(times):.4f}, 95th percentile {np.percentile(times, 95):.4f}")

val: int8 accuracy 0.8822 against full 0.8781; macro F1 0.8679 against 0.8643; labels changed 2 of 484
test: int8 accuracy 0.8285 against full 0.8347; macro F1 0.8137 against 0.8215; labels changed 3 of 484
int8 single-thread latency in seconds per sentence: median 0.0909, 95th percentile 0.1698


**Conclusion.** The int8 model scores 82.85% test accuracy and 0.8137 macro F1 against 83.47% and 0.8215 for the full model, changing 3 of 484 test labels; on validation it changes 2 labels and happens to gain (88.22% against 87.81%). The cost of quantisation is about half a point of accuracy, three sentences on a test split where one sentence is worth 0.21 points. A sentence takes 86 ms at the median and 163 ms at the 95th percentile on one Kaggle CPU thread; Render's free tier has a fraction of a CPU, so expect it to be slower there.

## Parity sentences

Twenty sentences written for this project (no dataset text) are scored with the int8 model and stored with their probabilities. The app's tests load the same model and must reproduce these numbers within 1e-4, which proves the app's tokenizer and runtime match this notebook's.

In [6]:
parity_probs = int8_probabilities(PARITY)
parity = [{"text": t, "probabilities": [round(float(v), 8) for v in p], "label": LABELS[int(p.argmax())]} for t, p in zip(PARITY, parity_probs)]
for row in parity:
    print(f"{row['label']:>8}  {max(row['probabilities']):.3f}  {row['text']}")

positive  0.994  Quarterly revenue rose 12 percent, beating analyst expectations.
negative  0.979  The company cut its full-year profit forecast after weak holiday sales.
 neutral  0.996  The board will publish the interim report on 14 August.
positive  0.727  The lender set aside more money for bad loans as defaults climbed.
positive  0.995  Operating margin improved as raw material costs eased.
 neutral  0.994  Shares were little changed after the annual meeting date was confirmed.
negative  0.991  Net sales decreased to 210 million euros from 245 million euros a year earlier.
positive  0.934  The new contract is expected to add about 40 million dollars to annual sales.
negative  0.810  Credit rating agencies downgraded the utility to junk status.
 neutral  0.919  Shareholders approved the merger at an extraordinary general meeting.
positive  0.996  Profit before taxes more than doubled thanks to strong demand in Asia.
negative  0.978  The retailer plans to close 150 stores and cut 3

**Conclusion.** The 20 sentences written for this project mostly read as intended: clear gains come out positive, cuts and losses negative, procedural statements neutral. Three misses are instructive: 'The lender set aside more money for bad loans as defaults climbed' is called positive (0.73), apparently reading the extra provisions as good news; the tenth consecutive dividend raise is called neutral (0.69); and the buyback is positive only at 0.52. These probabilities are what the app's parity test must reproduce.

## The serving bundle

Writes every file of the bundle contract: the tokenizer, labels, metrics (notebook 4's and 6's, with the int8 results added and the benchmark rows gathered in one list), chart data, the walk-forward sessions with missing values as JSON `null`, the parity sentences and the library versions. Sizes are printed; nothing in the bundle carries dataset text.

In [7]:
shutil.copy(model_dir / "tokenizer.json", ART / "tokenizer.json")
write("labels.json", LABELS)
sentiment_out = {k: v for k, v in sentiment.items() if k != "benchmarks"}
sentiment_out["int8"] = int8_result
market_out = {k: v for k, v in market.items() if k != "benchmarks"}
write("metrics.json", {"sentiment": sentiment_out, "market": market_out, "benchmarks": sentiment["benchmarks"] + market["benchmarks"]})
CONTRASTS = ["sentiment_classifier", "sentiment_returns", "returns_only", "previous_direction", "always_up"]
records = []
for row in sessions.to_dict("records"):
    in_test = bool(row["in_test"])
    up = {name: (None if pd.isna(row[f"up_{name}"]) else bool(row[f"up_{name}"])) for name in ["sentiment"] + CONTRASTS}
    actual_up = bool(row["target"] > 0)
    records.append({"date": pd.Timestamp(row["date"]).date().isoformat(), "count": int(row["count"]), "net_tone": row["net_tone"], "share_pos": row["share_pos"], "share_neg": row["share_neg"],
                    "target": row["target"], "in_test": in_test, "pred_return": row["score_sentiment"] if in_test else None, "pred_up": up["sentiment"], "actual_up": actual_up,
                    "correct": None if up["sentiment"] is None else up["sentiment"] == actual_up, "contrasts": {name: up[name] for name in CONTRASTS}})
write("sessions.json", records)
tested = sessions[sessions["in_test"]].set_index("date")
hit = (tested["up_sentiment"] == (tested["target"] > 0)).astype(float)
names_map = {"finbert": "Fine-tuned FinBERT", "baseline": "TF-IDF + Logistic Regression", "prosus": "ProsusAI/finbert (trained on PhraseBank)", **{k: v["label"] for k, v in market["models"].items()}}
charts = {
    "model_names": names_map,
    "rolling": {"window": 60, "dates": [d.date().isoformat() for d in tested.index], "accuracy": hit.rolling(60, min_periods=60).mean().round(6).tolist(), "always_up": (tested["target"] > 0).astype(float).rolling(60, min_periods=60).mean().round(6).tolist()},
    "tone": {"dates": [pd.Timestamp(d).date().isoformat() for d in sessions["date"]], "net_tone_r5": sessions["net_tone"].rolling(5, min_periods=1).mean().round(6).tolist(), "target_r5": sessions["target"].rolling(5, min_periods=1).mean().round(6).tolist()},
}
write("charts.json", charts)
write("parity.json", parity)
write("versions.json", {"python": platform.python_version(), "torch": torch.__version__, "transformers": transformers.__version__, "tokenizers": tokenizers.__version__, "onnx": onnx.__version__, "onnxruntime": ort.__version__, "numpy": np.__version__, "pandas": pd.__version__, "sklearn": sklearn.__version__})
for path in sorted(ART.iterdir()):
    print(f"artifacts/{path.name}: {path.stat().st_size / 1e6:.2f} MB")
print(f"elapsed {(time.time() - START) / 60:.1f} minutes")

artifacts/charts.json: 0.03 MB
artifacts/finbert-int8.onnx: 110.52 MB
artifacts/labels.json: 0.00 MB
artifacts/metrics.json: 0.01 MB
artifacts/parity.json: 0.00 MB
artifacts/sessions.json: 0.23 MB
artifacts/tokenizer.json: 0.71 MB
artifacts/versions.json: 0.00 MB
elapsed 6.2 minutes


**Conclusion.** All eight bundle files are written, 111.5 MB in total and almost all of it the int8 model; the JSON files carry metrics, chart series and session aggregates but no dataset text.